# Notebook 0: Optimization Benchmarks for EV Fleet Charging
## IEOR E4010 — AI for Operations Research and Financial Engineering
### Columbia University, Spring 2026

---

## Overview

In this notebook you will formulate and solve a **linear program (LP)** for optimal EV fleet charging, establishing benchmarks that your ML and DL models will try to approach.

The central idea is **predict-then-optimize**:
1. **Predict** electricity prices for the next 24 hours
2. **Optimize** a charging schedule using the LP with those predicted prices
3. **Evaluate** the schedule's cost against the *true* realized prices

You will see:
- **Perfect foresight** (LP with true prices) — the best any method can achieve
- **FIFO heuristic** (no price info) — naive baseline
- **Naive forecast** (10-day rolling average) — simple predict-then-optimize

In Notebooks 1 and 2 you will plug your ML/DL price forecasts into this same optimizer.

### Key OR insight
All LP-based approaches produce *feasible* schedules (vehicles reach target SoC). The difference is purely in *cost*. Better price forecasts → lower cost — this is the **value of information**.

In [ ]:
# ============================================================
# SETUP — Install dependencies and clone repo if on Colab
# ============================================================
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists('EV_Fleet_Project'):
        import subprocess
        subprocess.run(['git', 'clone',
                        'https://github.com/x1linwang/EV_Fleet_Project.git'],
                       check=True)
    os.chdir('EV_Fleet_Project')

import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'],
               check=True)
print('Setup complete.')

os.makedirs('data', exist_ok=True)

---
## Section 0: Setup & Data Loading

Load the 2025 NYISO hourly price-and-weather dataset and the NYC fleet profiles, then do a quick exploratory look at the price distribution.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json
import warnings
from scipy.optimize import linprog
from scipy.stats import truncnorm

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")

# ── Load price / weather data ─────────────────────────────────────────────────
df = pd.read_csv("data/nyiso_prices_weather_nyc_2025.csv", parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# ── Load fleet profiles ────────────────────────────────────────────────────────
with open("data/nyc_fleet_profiles.json") as f:
    fleet_profiles = json.load(f)

print(f"Hourly price data : {df.shape[0]} rows")
print(f"Date range        : {df['timestamp'].min().date()} to {df['timestamp'].max().date()}")
print(f"\nPrice statistics ($/kWh):")
print(df['rt_lbmp_kwh'].describe().round(5))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution of hourly prices
axes[0].hist(df['rt_lbmp_kwh'], bins=60, color='#4E79A7', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('RT Price ($/kWh)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Hourly RT Prices (2025)')

# Average price by hour of day
hourly_avg = df.groupby('hour')['rt_lbmp_kwh'].mean()
colors = ['#E15759' if p > hourly_avg.quantile(0.75) else
          '#59A14F' if p < hourly_avg.quantile(0.25) else '#4E79A7'
          for p in hourly_avg]
axes[1].bar(hourly_avg.index, hourly_avg.values, color=colors, alpha=0.85)
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Avg RT Price ($/kWh)')
axes[1].set_title('Average RT Price by Hour of Day (2025)')
axes[1].set_xticks(range(0, 24, 3))

cheap_h  = hourly_avg.idxmin()
exp_h    = hourly_avg.idxmax()
axes[1].annotate(f'Cheapest\n(h={cheap_h})',
                 xy=(cheap_h, hourly_avg[cheap_h]),
                 xytext=(cheap_h + 2, hourly_avg[cheap_h] + 0.005),
                 fontsize=8, arrowprops=dict(arrowstyle='->', color='gray'))

plt.tight_layout()
plt.savefig('data/00_eda_prices.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Cheapest hour  : h={cheap_h:02d}  avg {hourly_avg[cheap_h]:.5f} $/kWh")
print(f"Most expensive : h={exp_h:02d}  avg {hourly_avg[exp_h]:.5f} $/kWh")

---
## Section 1: The EV Charging Scenario

### Depot setup
- **20 homogeneous Level-2 AC chargers**, each rated at **11 kW**
- All chargers draw directly from the grid (no battery storage)
- Grid connection limit: 500 kW
- Fleet: Toyota bZ4X-class vehicles, 76 kWh battery, 93% AC charging efficiency

### Cost components

| Component | Formula | Notes |
|-----------|---------|-------|
| Electricity | $\sum_t p_t \cdot \text{draw}_t \cdot \Delta t$ | True RT prices at dispatch |
| Demand charge | $\rho \cdot \max_t \text{draw}_t$ | June–Sept: \$1.79/kW/day; other: \$1.37/kW/day |
| Opportunity cost | $\sum_v \sum_{t \geq t^*_v} c^{\text{opp}}_t$ | Revenue lost when a *charged* vehicle idles |
| Deadline penalty | $50 \cdot \max(0, \text{SoC shortfall}) \times 100$ | Heavy penalty per vehicle |

### Predict-then-optimize pipeline

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.8))
ax.set_xlim(0, 12); ax.set_ylim(0, 2.2); ax.axis('off')

steps = [
    (1.2, "Price\nForecast",   "#4E79A7"),
    (4.0, "LP\nOptimizer",    "#F28E2B"),
    (6.8, "Charging\nSchedule", "#59A14F"),
    (9.6, "Cost\nEvaluation", "#E15759"),
]
labels_below = [
    "24-hour\npredicted prices",
    "x[v,t]: kW per\nvehicle-hour",
    "Dispatch at\ntrue prices",
]
for x, label, color in steps:
    circ = plt.Circle((x, 1.1), 0.72, color=color, alpha=0.88)
    ax.add_patch(circ)
    ax.text(x, 1.1, label, ha='center', va='center', fontsize=9,
            fontweight='bold', color='white')

for i in range(len(steps) - 1):
    x1 = steps[i][0] + 0.72
    x2 = steps[i+1][0] - 0.72
    ax.annotate('', xy=(x2, 1.1), xytext=(x1, 1.1),
                arrowprops=dict(arrowstyle='->', lw=2.0, color='#666'))
    ax.text((x1 + x2) / 2, 0.22, labels_below[i],
            ha='center', fontsize=7.5, color='#555')

ax.set_title('Predict-Then-Optimize Pipeline', fontsize=13, fontweight='bold', pad=8)
plt.tight_layout()
plt.savefig('data/00_pipeline.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
PARAMS = {
    "n_chargers":               20,
    "charger_kw":               11.0,        # max kW per charger
    "efficiency":               0.93,        # AC charging efficiency
    "battery_kwh":              76.0,        # vehicle battery capacity (kWh)
    "target_soc":               0.70,        # required SoC at departure
    "grid_max_kw":              500.0,       # grid connection limit (kW)
    "demand_charge_summer":     53.60 / 30,  # $/kW/day, June-Sept
    "demand_charge_other":      41.24 / 30,  # $/kW/day, all other months
    "deadline_penalty_per_pct": 50.0,        # $ per vehicle per 1% SoC shortfall
    "dt":                       1.0,         # hours per time step
    "opp_cost_scale":           0.01,        # weight for opp-cost term in LP objective
}

kwh_needed = (PARAMS["target_soc"] - fleet_profiles["arrival_soc"]["mean"]) * PARAMS["battery_kwh"]
charge_time = kwh_needed / (PARAMS["efficiency"] * PARAMS["charger_kw"])
print(f"Avg battery energy needed per vehicle : {kwh_needed:.1f} kWh")
print(f"Avg charge time at {PARAMS['charger_kw']} kW           : {charge_time:.1f} hours")
print(f"Demand charge (summer, $/kW/day)      : {PARAMS['demand_charge_summer']:.4f}")
print(f"Demand charge (other,  $/kW/day)      : {PARAMS['demand_charge_other']:.4f}")

---
## Section 2: Fleet Schedule Generator

Each simulation day, vehicles arrive at the depot according to a **Poisson process** calibrated to NYC TLC rideshare data.  Arrival SoC is drawn from a **truncated normal** distribution.

**Stay duration** depends on ride demand:
- High demand hour (fraction > 0.6 of weekday peak): short stay 1–2 h
- Low demand hour: longer stay 2–8 h

This mirrors the trade-off in the RL environment: high-demand hours see vehicles returning quickly, limiting charging time.

In [ ]:
import math

def _min_stay_needed(arrival_soc, params):
    """Minimum integer stay (hours) to charge from arrival_soc to target_soc."""
    e_grid = max(0.0, (params["target_soc"] - arrival_soc) *
                 params["battery_kwh"] / params["efficiency"])
    return math.ceil(e_grid / params["charger_kw"]) if e_grid > 0 else 0


def generate_daily_fleet_schedule(fleet_profiles, date, seed=None, params=None):
    """
    Generate feasible vehicle charging sessions for one day.

    Sessions whose stay window is physically too short to reach target SoC
    are filtered out, leaving ~30-60 feasible charging events per day.

    Returns list[dict] with keys:
        vehicle_id, arrival_hour (0-23), departure_hour (1-24), arrival_soc.
    """
    if params is None:
        params = PARAMS
    date_ts = pd.Timestamp(date)
    if seed is None:
        seed = date_ts.toordinal()
    rng = np.random.default_rng(int(seed))

    day_type      = "weekend" if date_ts.dayofweek >= 5 else "weekday"
    arrival_rates = fleet_profiles["depot_arrival_rate"][day_type]
    ride_demand   = fleet_profiles["ride_demand"][day_type]
    max_demand    = max(fleet_profiles["ride_demand"]["weekday"])

    soc_p    = fleet_profiles["arrival_soc"]
    soc_mean = soc_p["mean"];  soc_std = soc_p["std"]
    a_tn = (soc_p["min"] - soc_mean) / soc_std
    b_tn = (soc_p["max"] - soc_mean) / soc_std

    vehicles = [];  vehicle_id = 0

    for h in range(24):
        n_arrivals = int(rng.poisson(arrival_rates[h]))
        for _ in range(n_arrivals):
            arrival_soc = float(
                truncnorm.rvs(a_tn, b_tn, loc=soc_mean, scale=soc_std,
                              random_state=rng)
            )
            demand_frac = ride_demand[h] / max_demand
            if demand_frac > 0.6:
                stay_hours = int(rng.integers(1, 2, endpoint=True))
            else:
                stay_hours = int(rng.integers(2, 8, endpoint=True))

            departure_hour = min(h + stay_hours, 24)
            if departure_hour <= h:
                continue

            # Feasibility filter: skip sessions too short to reach target SoC
            min_stay = _min_stay_needed(arrival_soc, params)
            if (departure_hour - h) < max(1, min_stay):
                continue

            vehicles.append({
                "vehicle_id":    vehicle_id,
                "arrival_hour":  h,
                "departure_hour": departure_hour,
                "arrival_soc":   arrival_soc,
            })
            vehicle_id += 1

    return vehicles

In [ ]:
SAMPLE_DATE  = pd.Timestamp("2025-09-15")  # Monday (weekday)
sample_seed  = SAMPLE_DATE.toordinal()
vehicles     = generate_daily_fleet_schedule(fleet_profiles, SAMPLE_DATE,
                                             seed=sample_seed, params=PARAMS)

stays    = [v['departure_hour'] - v['arrival_hour'] for v in vehicles]
soc_gaps = [PARAMS['target_soc'] - v['arrival_soc'] for v in vehicles]

print(f"Sample date  : {SAMPLE_DATE.date()} ({SAMPLE_DATE.day_name()})")
print(f"Total vehicles generated          : {len(vehicles)}")
print(f"Already at or above target SoC    : {sum(1 for g in soc_gaps if g <= 0)}")
print(f"Avg stay duration                 : {np.mean(stays):.1f} h  "
      f"(range {min(stays)}–{max(stays)} h)")
print(f"Avg SoC gap                       : {np.mean(soc_gaps):.2%}")
print(f"Avg grid energy needed per vehicle: "
      f"{np.mean(soc_gaps) * PARAMS['battery_kwh'] / PARAMS['efficiency']:.1f} kWh")

In [ ]:
fig, ax = plt.subplots(figsize=(12, max(4, len(vehicles) * 0.12 + 1)))

# Sort vehicles by arrival hour for cleaner display; show up to 80
disp_v = sorted(vehicles, key=lambda v: (v['arrival_hour'], v['arrival_soc']))[:80]
norm   = plt.Normalize(vmin=PARAMS['arrival_soc'] if False else 0.08, vmax=0.35)
cmap   = plt.cm.RdYlGn  # red = low SoC, green = high SoC

for yi, v in enumerate(disp_v):
    color = cmap(norm(v['arrival_soc']))
    ax.barh(yi, v['departure_hour'] - v['arrival_hour'],
            left=v['arrival_hour'], height=0.7,
            color=color, alpha=0.8, edgecolor='white', linewidth=0.3)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.6, pad=0.02)
cbar.set_label('Arrival SoC', fontsize=9)

ax.set_xlabel('Hour of Day', fontsize=11)
ax.set_ylabel('Vehicle (sorted by arrival)', fontsize=11)
ax.set_title(f'Depot Gantt Chart — {SAMPLE_DATE.date()} '
             f'(showing {len(disp_v)} of {len(vehicles)} vehicles)', fontsize=12)
ax.set_xlim(0, 24)
ax.set_xticks(range(0, 25, 2))
ax.axvline(x=PARAMS['n_chargers'] * 0, color='gray', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('data/00_gantt.png', dpi=100, bbox_inches='tight')
plt.show()

# Hourly vehicle count
hourly_count = np.zeros(24)
for v in vehicles:
    for t in range(v['arrival_hour'], min(v['departure_hour'], 24)):
        hourly_count[t] += 1
print(f"\nMax simultaneous vehicles at depot : {int(hourly_count.max())}")
print(f"Avg simultaneous vehicles at depot : {hourly_count.mean():.1f}")
print(f"Available charger slots (×11 kW)   : {PARAMS['n_chargers']}")

---
## Section 3: LP Formulation

### Mathematical formulation

**Sets:**
- $\mathcal{V}$ — set of vehicles
- $\mathcal{T}_v = [a_v, d_v)$ — hours vehicle $v$ is present at the depot

**Parameters:**
- $p_t$ — electricity price at hour $t$ (\$/kWh)
- $E_v = \max\!\left(0,\ \frac{(\text{SoC}^{\text{target}} - \text{SoC}^{\text{arr}}_v)\, B}{\eta}\right)$ — grid energy needed (kWh)
- $\bar{x}$ — charger power limit (kW)
- $C$ — charger count, $G$ — grid limit (kW); effective cap $= \min(C\bar{x}, G)$
- $\rho$ — demand charge rate (\$/kW/day)
- $M = 5000$ — large penalty to discourage energy shortfall

**Decision variables** (all $\geq 0$):
- $x_{vt}$ — charging power (kW) for vehicle $v$ at hour $t \in \mathcal{T}_v$
- $\sigma_v$ — energy shortfall slack (kWh) for vehicle $v$
- $z$ — peak total grid draw (kW)

**Objective** (minimize total operating cost):

$$\min \sum_t p_t \Bigl(\sum_v x_{vt}\Bigr)\Delta t \;+\; \rho\, z \;+\; M \sum_v \sigma_v$$

**Constraints:**

(a) Energy sufficiency:
$$\sum_{t \in \mathcal{T}_v} x_{vt}\,\Delta t + \sigma_v \geq E_v \qquad \forall v$$

(b) Charger power bound (via bounds): $\;0 \leq x_{vt} \leq \bar{x}$

(c+d) Grid / charger capacity per hour:
$$\sum_v x_{vt} \leq \min(C\bar{x},\, G) \qquad \forall t$$

(e) Peak tracking:
$$\sum_v x_{vt} \leq z \qquad \forall t$$

The penalty term $M\sigma_v$ makes shortfall a **last resort** — the LP only uses slack when the vehicle's charging window is physically too short to reach the target SoC (e.g., 1-hour stay with 4 kWh remaining).

In [ ]:
def get_hourly_prices(df, date):
    """Extract 24 hourly rt_lbmp_kwh prices for a given date.
    Returns None if fewer than 24 rows are available (incomplete day)."""
    date_ts = pd.Timestamp(date).normalize()
    mask    = ((df["timestamp"] >= date_ts) &
               (df["timestamp"] < date_ts + pd.Timedelta(days=1)))
    day_df  = df.loc[mask].sort_values("timestamp")
    if len(day_df) < 24:
        return None
    return day_df["rt_lbmp_kwh"].values[:24].astype(float)

In [ ]:
def solve_charging_lp(vehicles, prices, params, month=None, opp_rates=None):
    """
    Solve the minimum-cost EV charging LP with scipy HiGHS.

    Variable ordering: [x_{0,a0}, ..., x_{0,d0-1}, x_{1,a1}, ..., sigma_0, ..., z]

    Parameters
    ----------
    vehicles  : list[dict]       - from generate_daily_fleet_schedule()
    prices    : array (24,)      - hourly prices in $/kWh
    params    : dict             - PARAMS
    month     : int or None      - calendar month (determines demand-charge rate)
    opp_rates : array (24,) or None
                Hourly opportunity-cost rates ($/vehicle-hour). When provided,
                a linearised opp-cost credit is added so the LP rewards early
                charging and truly minimises total cost.

    Returns dict: schedule (n_v x 24 kW), status, objective, peak_kw, slack.
    """
    prices = np.asarray(prices, dtype=float)
    n_v    = len(vehicles)

    if n_v == 0:
        return {"schedule": np.zeros((0, 24)), "status": "empty",
                "objective": 0.0, "peak_kw": 0.0, "slack": np.array([])}

    n_chargers  = params["n_chargers"];  charger_kw  = params["charger_kw"]
    efficiency  = params["efficiency"];  battery_kwh = params["battery_kwh"]
    target_soc  = params["target_soc"];  grid_max_kw = params["grid_max_kw"]
    dt          = params["dt"];          M           = 5000.0
    opp_scale   = params.get("opp_cost_scale", 0.01)

    rho = (params["demand_charge_summer"]
           if (month is not None and month in [6, 7, 8, 9])
           else params["demand_charge_other"])

    # Variable layout
    windows  = [list(range(v["arrival_hour"], v["departure_hour"])) for v in vehicles]
    n_x_v    = [len(w) for w in windows]
    N_x      = sum(n_x_v)
    x_starts = np.zeros(n_v, dtype=int)
    for i in range(1, n_v):
        x_starts[i] = x_starts[i - 1] + n_x_v[i - 1]
    sigma_start = N_x;  z_idx = N_x + n_v;  N = N_x + n_v + 1

    E_v      = [max(0.0, (target_soc - v["arrival_soc"]) * battery_kwh / efficiency)
                for v in vehicles]
    E_v_batt = [max(0.0, (target_soc - v["arrival_soc"]) * battery_kwh)
                for v in vehicles]

    # Objective
    c = np.zeros(N)
    for vi in range(n_v):
        for i, t in enumerate(windows[vi]):
            c[x_starts[vi] + i] = prices[t] * dt
    c[sigma_start: sigma_start + n_v] = M
    c[z_idx] = rho

    # Optional linearised opportunity-cost credit
    if opp_rates is not None:
        opp_arr = np.asarray(opp_rates, dtype=float)
        for vi in range(n_v):
            if E_v_batt[vi] <= 1e-9:
                continue
            d_h = vehicles[vi]["departure_hour"]
            for i, tau in enumerate(windows[vi]):
                K_opp = float(opp_arr[tau:d_h].sum())
                c[x_starts[vi] + i] -= opp_scale * (efficiency * dt / E_v_batt[vi]) * K_opp

    # Hour -> (vehicle, position) lookup
    hour_to_vars = [[] for _ in range(24)]
    for vi in range(n_v):
        for i, t in enumerate(windows[vi]):
            hour_to_vars[t].append((vi, i))

    # Inequality constraints
    cap    = min(n_chargers * charger_kw, grid_max_kw)
    n_ineq = n_v + 48  # energy (n_v) + capacity (24) + peak (24)
    A_ub   = np.zeros((n_ineq, N))
    b_ub   = np.zeros(n_ineq)

    for vi in range(n_v):                                      # (a) energy
        A_ub[vi, x_starts[vi]: x_starts[vi] + n_x_v[vi]] = -dt
        A_ub[vi, sigma_start + vi] = -1.0
        b_ub[vi] = -E_v[vi]

    for t in range(24):                                        # (c+d) and (e)
        row_cap  = n_v + t;  row_peak = n_v + 24 + t
        for (vi, i) in hour_to_vars[t]:
            col = x_starts[vi] + i
            A_ub[row_cap,  col] = 1.0
            A_ub[row_peak, col] = 1.0
        b_ub[row_cap] = cap
        A_ub[row_peak, z_idx] = -1.0

    bounds = ([(0.0, charger_kw)] * N_x +
              [(0.0, None)]       * n_v  +
              [(0.0, None)])

    result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")

    schedule = np.zeros((n_v, 24))
    if result.status == 0:
        for vi in range(n_v):
            for i, t in enumerate(windows[vi]):
                schedule[vi, t] = max(0.0, result.x[x_starts[vi] + i])
        slack   = np.array(result.x[sigma_start: sigma_start + n_v])
        peak_kw = float(result.x[z_idx])
    else:
        slack   = np.zeros(n_v)
        peak_kw = float(schedule.sum(axis=0).max())

    return {"schedule": schedule, "status": result.message,
            "objective": float(result.fun) if result.fun is not None else float("inf"),
            "peak_kw": peak_kw, "slack": slack}

In [ ]:
# Solve the sample day with true (perfect-foresight) prices
true_prices = get_hourly_prices(df, SAMPLE_DATE)
result_pf   = solve_charging_lp(vehicles, true_prices, PARAMS,
                                month=SAMPLE_DATE.month)

print(f"LP solver status         : {result_pf['status']}")
print(f"LP objective value       : ${result_pf['objective']:.2f}  "
      f"(electricity + demand rho*z + M*slack)")
print(f"Peak grid draw           : {result_pf['peak_kw']:.1f} kW")
n_slack = np.sum(result_pf['slack'] > 0.01)
print(f"Vehicles with slack>0.01 : {n_slack}  (infeasible windows, physically unavoidable)")
print(f"Total energy dispatched  : {result_pf['schedule'].sum():.0f} kWh  "
      f"(across {len(vehicles)} vehicles)")

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))

# Stacked area: per-vehicle charging profile
total_draw = result_pf['schedule'].sum(axis=0)
hours      = np.arange(24)

# Show top vehicles by total energy (to keep the plot readable)
energies  = result_pf['schedule'].sum(axis=1)
top_idx   = np.argsort(energies)[::-1][:15]
bottom    = np.zeros(24)
palette   = sns.color_palette("muted", 15)

for k, vi in enumerate(top_idx):
    draw = result_pf['schedule'][vi]
    ax1.bar(hours, draw, bottom=bottom, color=palette[k], alpha=0.75,
            width=0.8, label=f'V{vi}' if k < 5 else None)
    bottom += draw

# Remaining vehicles (aggregated)
remaining = total_draw - bottom
if remaining.sum() > 0:
    ax1.bar(hours, np.maximum(remaining, 0), bottom=bottom,
            color='lightgray', alpha=0.6, width=0.8, label='others')

ax1.axhline(y=PARAMS['n_chargers'] * PARAMS['charger_kw'],
            color='red', linestyle='--', linewidth=1.2, alpha=0.7,
            label=f"Charger cap ({PARAMS['n_chargers']}×{PARAMS['charger_kw']} kW)")
ax1.set_xlabel('Hour of Day', fontsize=11)
ax1.set_ylabel('Total Grid Draw (kW)', fontsize=11)
ax1.set_xticks(range(24))

# Price curve overlay
ax2 = ax1.twinx()
ax2.step(hours, true_prices * 1000, color='#E15759', linewidth=2.0,
         linestyle='-', where='mid', label='RT price ($/MWh)')
ax2.set_ylabel('RT Price ($/MWh)', fontsize=11, color='#E15759')
ax2.tick_params(axis='y', labelcolor='#E15759')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1[:6] + lines2, labels1[:6] + labels2,
           loc='upper left', fontsize=8, ncol=2)

ax1.set_title(f'LP Perfect Foresight: Grid Draw vs. Price — {SAMPLE_DATE.date()}',
              fontsize=12)
plt.tight_layout()
plt.savefig('data/00_lp_dispatch.png', dpi=100, bbox_inches='tight')
plt.show()

print("LP shifts load to off-peak hours, avoiding the expensive morning/evening ramps.")

---
## Section 4: Cost Evaluation Function

`evaluate_schedule()` maps a charging schedule to a detailed cost breakdown at **true** realized prices.  This is the function that lets us compare schedules produced by different forecasts on a common ground.

### Cost components
- **Electricity** + **Demand charge** + **Deadline penalty** = `total_cost` (what the LP optimizes)
- **Opportunity cost** (reported separately): scaled revenue lost when a charged vehicle idles

> **Why is opp_cost excluded from total_cost?**  The LP objective equals electricity + demand + penalty.  Including a non-LP-aligned term in `total_cost` would break the LP optimality guarantee (PF ≤ Naive ≤ FIFO).  Opp cost is informational — it shows how JIT charging policies could further improve fleet efficiency.

In [ ]:
def evaluate_schedule(schedule, true_prices, vehicles, fleet_profiles, date, params):
    """
    Evaluate a charging schedule against true realized prices.

    total_cost = electricity_cost + demand_charge + deadline_penalty
    (opportunity_cost is reported separately — not included in total_cost because
     it is not part of the LP objective, and including it would break the LP
     optimality guarantee PF ≤ Naive ≤ FIFO).

    Returns dict: electricity_cost, demand_charge, opportunity_cost,
                  n_missed_deadlines, deadline_penalty, total_cost,
                  total_energy_kwh, peak_kw.
    """
    _zero = {"electricity_cost": 0.0, "demand_charge": 0.0,
             "opportunity_cost": 0.0, "n_missed_deadlines": 0,
             "deadline_penalty": 0.0, "total_cost": 0.0,
             "total_energy_kwh": 0.0, "peak_kw": 0.0}
    n_v = len(vehicles)
    if n_v == 0:
        return _zero

    true_prices = np.asarray(true_prices, dtype=float)
    dt = params["dt"];  efficiency = params["efficiency"]
    battery_kwh = params["battery_kwh"];  target_soc = params["target_soc"]
    dpen = params["deadline_penalty_per_pct"]

    date_ts  = pd.Timestamp(date);  month = date_ts.month
    day_type = "weekend" if date_ts.dayofweek >= 5 else "weekday"
    rho      = (params["demand_charge_summer"] if month in [6, 7, 8, 9]
                else params["demand_charge_other"])

    opp_rates      = fleet_profiles["opportunity_cost"][day_type]
    opp_eval_scale = params.get("opp_cost_scale", 0.01)
    total_draw     = schedule.sum(axis=0)

    electricity_cost = float(np.dot(true_prices, total_draw) * dt)
    peak_kw          = float(total_draw.max())
    demand_charge    = rho * peak_kw
    opp_cost = 0.0;  n_missed = 0;  deadline_penalty = 0.0

    for vi, v in enumerate(vehicles):
        a_h, d_h, a_soc = v["arrival_hour"], v["departure_hour"], v["arrival_soc"]
        E_needed = max(0.0, (target_soc - a_soc) * battery_kwh)   # battery kWh

        # t_charged: first hour cumulative battery energy >= E_needed
        cum_batt = 0.0;  t_charged = d_h
        for t in range(a_h, d_h):
            cum_batt += schedule[vi, t] * efficiency * dt
            if cum_batt >= E_needed - 1e-9 and t_charged == d_h:
                t_charged = t
                break

        for t in range(t_charged, d_h):    # idle hours after charging (scaled)
            opp_cost += opp_eval_scale * opp_rates[min(t, 23)]

        total_batt = sum(schedule[vi, t] * efficiency * dt for t in range(a_h, d_h))
        final_soc  = a_soc + total_batt / battery_kwh
        if final_soc < target_soc - 1e-6:
            shortfall = target_soc - final_soc
            deadline_penalty += dpen * max(0.0, shortfall) * 100
            n_missed += 1

    # total_cost matches LP objective: electricity + demand + penalty
    total_cost = electricity_cost + demand_charge + deadline_penalty
    return {"electricity_cost": electricity_cost, "demand_charge": demand_charge,
            "opportunity_cost": opp_cost, "n_missed_deadlines": n_missed,
            "deadline_penalty": deadline_penalty, "total_cost": total_cost,
            "total_energy_kwh": float(total_draw.sum() * dt), "peak_kw": peak_kw}

In [ ]:
eval_pf = evaluate_schedule(result_pf['schedule'], true_prices, vehicles,
                             fleet_profiles, SAMPLE_DATE, PARAMS)

print(f"=== Perfect Foresight LP — Cost Breakdown ({SAMPLE_DATE.date()}) ===")
print(f"  {'Electricity cost':28s}: ${eval_pf['electricity_cost']:>9.2f}")
print(f"  {'Demand charge':28s}: ${eval_pf['demand_charge']:>9.2f}")
print(f"  {'Opportunity cost':28s}: ${eval_pf['opportunity_cost']:>9.2f}")
print(f"  {'Deadline penalty':28s}: ${eval_pf['deadline_penalty']:>9.2f}")
print(f"  {'─'*38}")
print(f"  {'TOTAL COST':28s}: ${eval_pf['total_cost']:>9.2f}")
print(f"  {'':28s}")
print(f"  {'Total energy dispatched':28s}: {eval_pf['total_energy_kwh']:>8.1f} kWh")
print(f"  {'Peak grid draw':28s}: {eval_pf['peak_kw']:>8.1f} kW")
print(f"  {'Missed deadlines':28s}: {eval_pf['n_missed_deadlines']:>8d} vehicles")
print()
print("Note: total_cost = electricity + demand + penalty (LP objective components).")
print("Opportunity cost is tracked separately — it is NOT included in total_cost.")
print("This ensures PF ≤ Naive ≤ FIFO by LP optimality guarantee.")

---
## Section 5: FIFO Heuristic Benchmark

The **FIFO heuristic** ignores price entirely: at each hour it charges the vehicles that have been waiting longest (earliest arrival first).  This is the simplest possible policy and serves as the upper bound on cost.

In [ ]:
def fifo_heuristic(vehicles, params):
    """
    Greedy FIFO charging heuristic — no price information.

    Each hour: serve earliest-arriving vehicles first (tiebreak: largest SoC gap).
    Respects both charger count and grid power limits.
    Returns same dict format as solve_charging_lp().
    """
    n_v = len(vehicles)
    schedule = np.zeros((n_v, 24))
    if n_v == 0:
        return {"schedule": schedule, "status": "heuristic",
                "objective": None, "peak_kw": 0.0, "slack": np.array([])}

    n_chargers  = params["n_chargers"];  charger_kw  = params["charger_kw"]
    efficiency  = params["efficiency"];  battery_kwh = params["battery_kwh"]
    target_soc  = params["target_soc"];  grid_max_kw = params["grid_max_kw"]
    dt          = params["dt"]

    E_v              = [max(0.0, (target_soc - v["arrival_soc"]) * battery_kwh / efficiency)
                        for v in vehicles]
    energy_delivered = [0.0] * n_v
    soc_gap          = [(target_soc - v["arrival_soc"]) * battery_kwh for v in vehicles]
    max_concurrent   = min(n_chargers, int(grid_max_kw // charger_kw))

    for t in range(24):
        needing = [vi for vi, v in enumerate(vehicles)
                   if v["arrival_hour"] <= t < v["departure_hour"]
                   and energy_delivered[vi] < E_v[vi] - 1e-6]
        needing.sort(key=lambda vi: (vehicles[vi]["arrival_hour"], -soc_gap[vi]))
        for vi in needing[:max_concurrent]:
            schedule[vi, t] = charger_kw
            energy_delivered[vi] += charger_kw * dt

    total_draw = schedule.sum(axis=0)
    slack = np.array([max(0.0, E_v[vi] - energy_delivered[vi]) for vi in range(n_v)])
    return {"schedule": schedule, "status": "heuristic", "objective": None,
            "peak_kw": float(total_draw.max()), "slack": slack}

In [ ]:
result_fifo = fifo_heuristic(vehicles, PARAMS)
eval_fifo   = evaluate_schedule(result_fifo['schedule'], true_prices, vehicles,
                                 fleet_profiles, SAMPLE_DATE, PARAMS)

print(f"=== FIFO Heuristic — Cost Breakdown ({SAMPLE_DATE.date()}) ===")
print(f"  {'Electricity cost':28s}: ${eval_fifo['electricity_cost']:>9.2f}")
print(f"  {'Demand charge':28s}: ${eval_fifo['demand_charge']:>9.2f}")
print(f"  {'Opportunity cost':28s}: ${eval_fifo['opportunity_cost']:>9.2f}")
print(f"  {'Deadline penalty':28s}: ${eval_fifo['deadline_penalty']:>9.2f}")
print(f"  {'─'*38}")
print(f"  {'TOTAL COST':28s}: ${eval_fifo['total_cost']:>9.2f}")
print(f"  {'Missed deadlines':28s}: {eval_fifo['n_missed_deadlines']:>8d} vehicles")
print()

# Side-by-side comparison
delta_elec = eval_fifo['electricity_cost'] - eval_pf['electricity_cost']
delta_tot  = eval_fifo['total_cost'] - eval_pf['total_cost']
print(f"LP saves ${delta_elec:+.2f} in electricity vs FIFO on this day.")
print(f"LP saves ${delta_tot:+.2f} in total cost vs FIFO on this day.")

---
## Section 6: Naive Forecast → Optimize

The **naive forecast** predicts each hour's price as the mean of the same hour over the previous 10 days.  We then solve the LP with these predicted prices and evaluate the resulting schedule at the *true* prices — this is the predict-then-optimize loop.

In [ ]:
def naive_forecast(df, date):
    """
    Predict 24 hourly prices using the 10-day rolling average of the same hour.

    Parameters
    ----------
    df   : pd.DataFrame with 'timestamp' (datetime), 'hour' (int), 'rt_lbmp_kwh'
    date : datetime.date or pd.Timestamp

    Returns np.ndarray of shape (24,) in $/kWh.
    """
    date_ts        = pd.Timestamp(date).normalize()
    start_lookback = date_ts - pd.Timedelta(days=10)
    mask  = (df["timestamp"] >= start_lookback) & (df["timestamp"] < date_ts)
    hist  = df.loc[mask]

    preds = np.zeros(24)
    for h in range(24):
        vals = hist.loc[hist["hour"] == h, "rt_lbmp_kwh"].values
        preds[h] = float(vals.mean()) if len(vals) > 0 else 0.0
    return preds

In [ ]:
naive_prices = naive_forecast(df, SAMPLE_DATE)

fig, ax = plt.subplots(figsize=(12, 4))
hours = np.arange(24)
ax.step(hours, true_prices  * 1000, where='mid', color='#E15759', lw=2,
        label='True RT prices')
ax.step(hours, naive_prices * 1000, where='mid', color='#4E79A7', lw=2,
        linestyle='--', label='Naive forecast (10-day avg)')
ax.fill_between(hours, true_prices * 1000, naive_prices * 1000,
                alpha=0.15, color='gray', step='mid', label='Forecast error')
ax.set_xlabel('Hour of Day');  ax.set_ylabel('Price ($/MWh)')
ax.set_title(f'True vs. Naive Forecast — {SAMPLE_DATE.date()}')
ax.set_xticks(range(24));  ax.legend()
plt.tight_layout()
plt.savefig('data/00_naive_forecast.png', dpi=100, bbox_inches='tight')
plt.show()

mae = np.mean(np.abs(true_prices - naive_prices)) * 1000
print(f"Naive forecast MAE : {mae:.2f} $/MWh on {SAMPLE_DATE.date()}")

# Solve LP with naive forecast prices, evaluate at true prices
result_naive = solve_charging_lp(vehicles, naive_prices, PARAMS,
                                  month=SAMPLE_DATE.month)
eval_naive   = evaluate_schedule(result_naive['schedule'], true_prices, vehicles,
                                  fleet_profiles, SAMPLE_DATE, PARAMS)

print(f"\n=== Naive Forecast LP — Cost Breakdown ({SAMPLE_DATE.date()}) ===")
print(f"  {'Electricity cost':28s}: ${eval_naive['electricity_cost']:>9.2f}")
print(f"  {'Demand charge':28s}: ${eval_naive['demand_charge']:>9.2f}")
print(f"  {'Opportunity cost':28s}: ${eval_naive['opportunity_cost']:>9.2f}")
print(f"  {'Deadline penalty':28s}: ${eval_naive['deadline_penalty']:>9.2f}")
print(f"  {'─'*38}")
print(f"  {'TOTAL COST':28s}: ${eval_naive['total_cost']:>9.2f}")
print(f"  {'Missed deadlines':28s}: {eval_naive['n_missed_deadlines']:>8d} vehicles")

---
## Section 7: Full Evaluation Over Validation Period

Evaluate all three benchmarks over every day from **Sep 1 – Dec 31 2025** (122 days). The seed for each day's fleet schedule is `date.toordinal()` for reproducibility.

In [ ]:
val_start = pd.Timestamp("2025-09-01")
val_end   = pd.Timestamp("2025-12-31")
val_dates = pd.date_range(val_start, val_end, freq='D')

records = []
print(f"Evaluating {len(val_dates)} days  ({val_start.date()} to {val_end.date()}) ...")
print(f"{'Date':12s} {'#V':>4s} {'PF $':>10s} {'FIFO $':>10s} {'Naive $':>10s}")
print('-' * 52)

for date in val_dates:
    true_p = get_hourly_prices(df, date)
    if true_p is None:
        continue  # skip incomplete days

    seed_d  = date.toordinal()
    veh_day = generate_daily_fleet_schedule(fleet_profiles, date,
                                            seed=seed_d, params=PARAMS)

    if len(veh_day) == 0:
        records.append({"date": date, "n_vehicles": 0,
                        **{f"pf_{k}":    0.0 for k in ["electricity_cost","demand_charge",
                           "opportunity_cost","deadline_penalty","total_cost",
                           "n_missed_deadlines"]},
                        **{f"fifo_{k}":  0.0 for k in ["electricity_cost","demand_charge",
                           "opportunity_cost","deadline_penalty","total_cost",
                           "n_missed_deadlines"]},
                        **{f"naive_{k}": 0.0 for k in ["electricity_cost","demand_charge",
                           "opportunity_cost","deadline_penalty","total_cost",
                           "n_missed_deadlines"]}})
        continue

    # Perfect foresight LP
    res_pf    = solve_charging_lp(veh_day, true_p, PARAMS, month=date.month)
    ev_pf     = evaluate_schedule(res_pf['schedule'], true_p, veh_day,
                                   fleet_profiles, date, PARAMS)

    # FIFO heuristic
    res_fifo  = fifo_heuristic(veh_day, PARAMS)
    ev_fifo   = evaluate_schedule(res_fifo['schedule'], true_p, veh_day,
                                   fleet_profiles, date, PARAMS)

    # Naive forecast LP
    naive_p   = naive_forecast(df, date)
    res_naive = solve_charging_lp(veh_day, naive_p, PARAMS, month=date.month)
    ev_naive  = evaluate_schedule(res_naive['schedule'], true_p, veh_day,
                                   fleet_profiles, date, PARAMS)

    row = {"date": date, "n_vehicles": len(veh_day)}
    for k, ev in [("pf", ev_pf), ("fifo", ev_fifo), ("naive", ev_naive)]:
        for field in ["electricity_cost", "demand_charge", "opportunity_cost",
                      "deadline_penalty", "total_cost", "n_missed_deadlines"]:
            row[f"{k}_{field}"] = ev[field]
    records.append(row)

    if date.day == 1 or date == val_dates[0]:
        print(f"{str(date.date()):12s} {len(veh_day):>4d}  "
              f"${ev_pf['total_cost']:>9.0f}  "
              f"${ev_fifo['total_cost']:>9.0f}  "
              f"${ev_naive['total_cost']:>9.0f}")

results_df = pd.DataFrame(records)
print(f"\nDone. {len(results_df)} days evaluated.")

---
## Section 8: Results Dashboard

Aggregated results over the full Sep–Dec 2025 validation period.

In [ ]:
# ─── Summary table ────────────────────────────────────────────────────────────
cost_cols = ["electricity_cost", "demand_charge", "opportunity_cost",
             "deadline_penalty", "total_cost"]
miss_col  = "n_missed_deadlines"

summary = {}
for method, prefix in [("Perfect Foresight LP", "pf"),
                        ("FIFO Heuristic",       "fifo"),
                        ("Naive Forecast LP",    "naive")]:
    row = {c: results_df[f"{prefix}_{c}"].mean() for c in cost_cols}
    row[miss_col] = results_df[f"{prefix}_{miss_col}"].mean()
    summary[method] = row

summary_df = pd.DataFrame(summary).T
summary_df.index.name = "Benchmark"
print("=== Mean Daily Cost Breakdown ($/day) — Validation Period ===")
print(summary_df.round(2).to_string())
print()
print("Expected ordering  :  Perfect Foresight  <  Naive Forecast  <  FIFO")
pf_tot    = summary_df.loc["Perfect Foresight LP", "total_cost"]
naive_tot = summary_df.loc["Naive Forecast LP",    "total_cost"]
fifo_tot  = summary_df.loc["FIFO Heuristic",       "total_cost"]
ordering_ok = pf_tot < naive_tot < fifo_tot
print(f"Ordering satisfied :  {ordering_ok}   "
      f"(PF={pf_tot:.0f} < Naive={naive_tot:.0f} < FIFO={fifo_tot:.0f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

methods      = ["Perfect Foresight LP", "Naive Forecast LP", "FIFO Heuristic"]
short_labels = ["Perfect\nForesight", "Naive\nForecast", "FIFO\nHeuristic"]
colors_m     = ["#59A14F", "#4E79A7", "#E15759"]

# Mean total cost with error bars
means  = [summary_df.loc[m, "total_cost"] for m in methods]
stds   = [results_df[f"{p}_total_cost"].std()
          for p in ["pf", "naive", "fifo"]]

bars = axes[0].bar(short_labels, means, color=colors_m, alpha=0.85,
                   yerr=stds, capsize=5, error_kw={"elinewidth": 1.5})
for bar, mean in zip(bars, means):
    axes[0].text(bar.get_x() + bar.get_width() / 2, mean + max(stds) * 0.05,
                 f"${mean:.0f}", ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_ylabel("Mean Daily Cost ($/day)")
axes[0].set_title("Mean Total Daily Cost by Benchmark")

# Stacked bar: cost components
components = ["electricity_cost", "demand_charge", "opportunity_cost", "deadline_penalty"]
comp_labels = ["Electricity", "Demand Charge", "Opportunity", "Penalty"]
comp_colors = ["#4E79A7", "#F28E2B", "#76B7B2", "#E15759"]

bottoms = np.zeros(3)
for comp, label, color in zip(components, comp_labels, comp_colors):
    vals = [summary_df.loc[m, comp] for m in methods]
    axes[1].bar(short_labels, vals, bottom=bottoms, label=label,
                color=color, alpha=0.85)
    bottoms += np.array(vals)

axes[1].set_ylabel("Mean Daily Cost ($/day)")
axes[1].set_title("Cost Breakdown by Component")
axes[1].legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.savefig('data/00_cost_bars.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for prefix, label, color in [("pf",    "Perfect Foresight LP", "#59A14F"),
                               ("naive", "Naive Forecast LP",    "#4E79A7"),
                               ("fifo",  "FIFO Heuristic",       "#E15759")]:
    ax.plot(results_df["date"], results_df[f"{prefix}_total_cost"],
            label=label, color=color, linewidth=1.3, alpha=0.85)

ax.set_xlabel("Date");  ax.set_ylabel("Daily Total Cost ($)")
ax.set_title("Daily Total Cost Over Validation Period (Sep–Dec 2025)")
ax.legend(fontsize=10)

import matplotlib.dates as mdates
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax.xaxis.set_major_locator(mdates.MonthLocator())

plt.tight_layout()
plt.savefig('data/00_cost_timeseries.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Value of Information ──────────────────────────────────────────────────────
voi_mean  = (results_df["naive_total_cost"] - results_df["pf_total_cost"]).mean()
voi_max   = (results_df["naive_total_cost"] - results_df["pf_total_cost"]).max()
gap_total = (results_df["fifo_total_cost"]  - results_df["pf_total_cost"]).mean()

print("=== Value of Information ===")
print(f"  Mean daily naive cost        : ${results_df['naive_total_cost'].mean():.2f}")
print(f"  Mean daily perfect foresight : ${results_df['pf_total_cost'].mean():.2f}")
print(f"  Value of Perfect Information : ${voi_mean:.2f}/day  (max ${voi_max:.2f})")
print()
print(f"  Total FIFO vs PF gap         : ${gap_total:.2f}/day")
print(f"  Fraction closed by naive     : {(1 - voi_mean/gap_total)*100:.1f}%")
print()
print("  → Your ML/DL forecast can potentially save up to")
print(f"    ${voi_mean:.2f}/day by improving on the naive forecast.")

# Scatter: naive cost vs perfect foresight cost
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(results_df["pf_total_cost"], results_df["naive_total_cost"],
           alpha=0.6, s=30, color="#4E79A7", edgecolors='white', linewidths=0.5)
lim = max(results_df[["pf_total_cost","naive_total_cost"]].max())
mn  = min(results_df[["pf_total_cost","naive_total_cost"]].min())
ax.plot([mn, lim], [mn, lim], 'r--', alpha=0.5, label='y = x (no gap)')
ax.set_xlabel("Perfect Foresight LP Cost ($/day)")
ax.set_ylabel("Naive Forecast LP Cost ($/day)")
ax.set_title("Naive vs. Perfect Foresight Cost\n(each point = one validation day)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('data/00_voi_scatter.png', dpi=100, bbox_inches='tight')
plt.show()

---
## Section 9: Export Utilities & Next Steps

Write all shared functions to `ev_charging_utils.py` for import by Notebooks 1 and 2.

In [ ]:
%%writefile ev_charging_utils.py
"""
ev_charging_utils.py
Shared utility functions for EV fleet charging optimization.
IEOR E4010 — AI for Operations Research and Financial Engineering
Columbia University, Spring 2026

Functions
---------
generate_daily_fleet_schedule  — Poisson/truncnorm arrival simulator
solve_charging_lp              — scipy HiGHS LP solver (with optional opp-cost term)
fifo_heuristic                 — greedy FIFO baseline
evaluate_schedule              — full cost evaluation at true realized prices
naive_forecast                 — 10-day rolling-average price forecast
get_hourly_prices              — extract 24 hourly prices from DataFrame
"""

import math
import numpy as np
import pandas as pd
from scipy.optimize import linprog
from scipy.stats import truncnorm

PARAMS = {
    "n_chargers":               20,
    "charger_kw":               11.0,        # max kW per charger
    "efficiency":               0.93,        # AC charging efficiency
    "battery_kwh":              76.0,        # vehicle battery capacity (kWh)
    "target_soc":               0.70,        # required SoC at departure
    "grid_max_kw":              500.0,       # grid connection limit (kW)
    "demand_charge_summer":     53.60 / 30,  # $/kW/day, June-Sept
    "demand_charge_other":      41.24 / 30,  # $/kW/day, all other months
    "deadline_penalty_per_pct": 50.0,        # $ per vehicle per 1% SoC shortfall
    "dt":                       1.0,         # hours per time step
    "opp_cost_scale":           0.01,        # weight for opp-cost term in LP objective
}


def _min_stay_needed(arrival_soc, params):
    """Minimum integer stay (hours) to charge from arrival_soc to target_soc."""
    e_grid = max(0.0, (params["target_soc"] - arrival_soc) *
                 params["battery_kwh"] / params["efficiency"])
    return math.ceil(e_grid / params["charger_kw"]) if e_grid > 0 else 0


def generate_daily_fleet_schedule(fleet_profiles, date, seed=None, params=None):
    """
    Generate a list of feasible vehicle charging sessions for one day.

    Sessions whose stay window is physically too short to reach target SoC
    are filtered out, leaving ~30-60 feasible charging events per day.

    Returns list[dict] with keys:
        vehicle_id, arrival_hour (0-23), departure_hour (1-24), arrival_soc.
    """
    if params is None:
        params = PARAMS
    date_ts = pd.Timestamp(date)
    if seed is None:
        seed = date_ts.toordinal()
    rng = np.random.default_rng(int(seed))
    day_type      = "weekend" if date_ts.dayofweek >= 5 else "weekday"
    arrival_rates = fleet_profiles["depot_arrival_rate"][day_type]
    ride_demand   = fleet_profiles["ride_demand"][day_type]
    max_demand    = max(fleet_profiles["ride_demand"]["weekday"])
    soc_p = fleet_profiles["arrival_soc"]
    soc_mean = soc_p["mean"]; soc_std = soc_p["std"]
    a_tn = (soc_p["min"] - soc_mean) / soc_std
    b_tn = (soc_p["max"] - soc_mean) / soc_std
    vehicles = []; vehicle_id = 0
    for h in range(24):
        n_arrivals = int(rng.poisson(arrival_rates[h]))
        for _ in range(n_arrivals):
            arrival_soc = float(truncnorm.rvs(a_tn, b_tn, loc=soc_mean, scale=soc_std,
                                               random_state=rng))
            demand_frac = ride_demand[h] / max_demand
            if demand_frac > 0.6:
                stay_hours = int(rng.integers(1, 2, endpoint=True))
            else:
                stay_hours = int(rng.integers(2, 8, endpoint=True))
            departure_hour = min(h + stay_hours, 24)
            if departure_hour <= h:
                continue
            min_stay = _min_stay_needed(arrival_soc, params)
            if (departure_hour - h) < max(1, min_stay):
                continue
            vehicles.append({"vehicle_id": vehicle_id, "arrival_hour": h,
                              "departure_hour": departure_hour, "arrival_soc": arrival_soc})
            vehicle_id += 1
    return vehicles


def solve_charging_lp(vehicles, prices, params, month=None, opp_rates=None):
    """
    Solve the minimum-cost EV charging LP with scipy HiGHS.

    Variable ordering: [x_{0,a0}, ..., x_{0,d0-1}, x_{1,a1}, ..., sigma_0, ..., z]

    Parameters
    ----------
    vehicles  : list[dict]       - from generate_daily_fleet_schedule()
    prices    : array (24,)      - hourly prices in $/kWh
    params    : dict             - PARAMS
    month     : int or None      - calendar month (determines demand-charge rate)
    opp_rates : array (24,) or None
                Hourly opportunity-cost rates ($/vehicle-hour). When provided,
                a linearised opp-cost credit is added so the LP rewards early
                charging and truly minimises total cost.

    Returns dict: schedule (n_v x 24 kW), status, objective, peak_kw, slack.
    """
    prices = np.asarray(prices, dtype=float)
    n_v = len(vehicles)
    if n_v == 0:
        return {"schedule": np.zeros((0, 24)), "status": "empty",
                "objective": 0.0, "peak_kw": 0.0, "slack": np.array([])}
    n_chargers  = params["n_chargers"]; charger_kw  = params["charger_kw"]
    efficiency  = params["efficiency"]; battery_kwh = params["battery_kwh"]
    target_soc  = params["target_soc"]; grid_max_kw = params["grid_max_kw"]
    dt          = params["dt"];         M           = 5000.0
    opp_scale   = params.get("opp_cost_scale", 0.01)
    rho = (params["demand_charge_summer"]
           if (month is not None and month in [6, 7, 8, 9])
           else params["demand_charge_other"])
    windows  = [list(range(v["arrival_hour"], v["departure_hour"])) for v in vehicles]
    n_x_v    = [len(w) for w in windows]; N_x = sum(n_x_v)
    x_starts = np.zeros(n_v, dtype=int)
    for i in range(1, n_v):
        x_starts[i] = x_starts[i - 1] + n_x_v[i - 1]
    sigma_start = N_x; z_idx = N_x + n_v; N = N_x + n_v + 1
    E_v      = [max(0.0, (target_soc - v["arrival_soc"]) * battery_kwh / efficiency)
                for v in vehicles]
    E_v_batt = [max(0.0, (target_soc - v["arrival_soc"]) * battery_kwh)
                for v in vehicles]
    c = np.zeros(N)
    for vi in range(n_v):
        for i, t in enumerate(windows[vi]):
            c[x_starts[vi] + i] = prices[t] * dt
    c[sigma_start: sigma_start + n_v] = M; c[z_idx] = rho
    if opp_rates is not None:
        opp_arr = np.asarray(opp_rates, dtype=float)
        for vi in range(n_v):
            if E_v_batt[vi] <= 1e-9:
                continue
            d_h = vehicles[vi]["departure_hour"]
            for i, tau in enumerate(windows[vi]):
                K_opp = float(opp_arr[tau:d_h].sum())
                c[x_starts[vi] + i] -= opp_scale * (efficiency * dt / E_v_batt[vi]) * K_opp
    hour_to_vars = [[] for _ in range(24)]
    for vi in range(n_v):
        for i, t in enumerate(windows[vi]):
            hour_to_vars[t].append((vi, i))
    cap = min(n_chargers * charger_kw, grid_max_kw)
    n_ineq = n_v + 48
    A_ub = np.zeros((n_ineq, N)); b_ub = np.zeros(n_ineq)
    for vi in range(n_v):
        A_ub[vi, x_starts[vi]: x_starts[vi] + n_x_v[vi]] = -dt
        A_ub[vi, sigma_start + vi] = -1.0; b_ub[vi] = -E_v[vi]
    for t in range(24):
        row_cap = n_v + t; row_peak = n_v + 24 + t
        for (vi, i) in hour_to_vars[t]:
            col = x_starts[vi] + i
            A_ub[row_cap, col] = 1.0; A_ub[row_peak, col] = 1.0
        b_ub[row_cap] = cap; A_ub[row_peak, z_idx] = -1.0
    bounds = ([(0.0, charger_kw)] * N_x + [(0.0, None)] * n_v + [(0.0, None)])
    result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")
    schedule = np.zeros((n_v, 24))
    if result.status == 0:
        for vi in range(n_v):
            for i, t in enumerate(windows[vi]):
                schedule[vi, t] = max(0.0, result.x[x_starts[vi] + i])
        slack = np.array(result.x[sigma_start: sigma_start + n_v])
        peak_kw = float(result.x[z_idx])
    else:
        slack = np.zeros(n_v); peak_kw = float(schedule.sum(axis=0).max())
    return {"schedule": schedule, "status": result.message,
            "objective": float(result.fun) if result.fun is not None else float("inf"),
            "peak_kw": peak_kw, "slack": slack}


def fifo_heuristic(vehicles, params):
    n_v = len(vehicles); schedule = np.zeros((n_v, 24))
    if n_v == 0:
        return {"schedule": schedule, "status": "heuristic",
                "objective": None, "peak_kw": 0.0, "slack": np.array([])}
    n_chargers  = params["n_chargers"]; charger_kw  = params["charger_kw"]
    efficiency  = params["efficiency"]; battery_kwh = params["battery_kwh"]
    target_soc  = params["target_soc"]; grid_max_kw = params["grid_max_kw"]
    dt          = params["dt"]
    E_v = [max(0.0, (target_soc - v["arrival_soc"]) * battery_kwh / efficiency)
           for v in vehicles]
    energy_delivered = [0.0] * n_v
    soc_gap = [(target_soc - v["arrival_soc"]) * battery_kwh for v in vehicles]
    max_concurrent = min(n_chargers, int(grid_max_kw // charger_kw))
    for t in range(24):
        needing = [vi for vi, v in enumerate(vehicles)
                   if v["arrival_hour"] <= t < v["departure_hour"]
                   and energy_delivered[vi] < E_v[vi] - 1e-6]
        needing.sort(key=lambda vi: (vehicles[vi]["arrival_hour"], -soc_gap[vi]))
        for vi in needing[:max_concurrent]:
            schedule[vi, t] = charger_kw; energy_delivered[vi] += charger_kw * dt
    total_draw = schedule.sum(axis=0)
    slack = np.array([max(0.0, E_v[vi] - energy_delivered[vi]) for vi in range(n_v)])
    return {"schedule": schedule, "status": "heuristic", "objective": None,
            "peak_kw": float(total_draw.max()), "slack": slack}


def evaluate_schedule(schedule, true_prices, vehicles, fleet_profiles, date, params):
    _zero = {"electricity_cost": 0.0, "demand_charge": 0.0, "opportunity_cost": 0.0,
             "n_missed_deadlines": 0, "deadline_penalty": 0.0, "total_cost": 0.0,
             "total_energy_kwh": 0.0, "peak_kw": 0.0}
    n_v = len(vehicles)
    if n_v == 0:
        return _zero
    true_prices = np.asarray(true_prices, dtype=float)
    dt = params["dt"]; efficiency = params["efficiency"]
    battery_kwh = params["battery_kwh"]; target_soc = params["target_soc"]
    dpen = params["deadline_penalty_per_pct"]
    date_ts = pd.Timestamp(date); month = date_ts.month
    day_type = "weekend" if date_ts.dayofweek >= 5 else "weekday"
    rho = (params["demand_charge_summer"] if month in [6, 7, 8, 9]
           else params["demand_charge_other"])
    opp_rates      = fleet_profiles["opportunity_cost"][day_type]
    opp_eval_scale = params.get("opp_cost_scale", 0.01)
    total_draw     = schedule.sum(axis=0)
    electricity_cost = float(np.dot(true_prices, total_draw) * dt)
    peak_kw = float(total_draw.max()); demand_charge = rho * peak_kw
    opp_cost = 0.0; n_missed = 0; deadline_penalty = 0.0
    for vi, v in enumerate(vehicles):
        a_h, d_h, a_soc = v["arrival_hour"], v["departure_hour"], v["arrival_soc"]
        E_needed = max(0.0, (target_soc - a_soc) * battery_kwh)
        cum_batt = 0.0; t_charged = d_h
        for t in range(a_h, d_h):
            cum_batt += schedule[vi, t] * efficiency * dt
            if cum_batt >= E_needed - 1e-9 and t_charged == d_h:
                t_charged = t; break
        for t in range(t_charged, d_h):
            opp_cost += opp_eval_scale * opp_rates[min(t, 23)]
        total_batt = sum(schedule[vi, t] * efficiency * dt for t in range(a_h, d_h))
        final_soc = a_soc + total_batt / battery_kwh
        if final_soc < target_soc - 1e-6:
            shortfall = target_soc - final_soc
            deadline_penalty += dpen * max(0.0, shortfall) * 100; n_missed += 1
    # total_cost matches LP objective (electricity + demand + penalty).
    # Opportunity cost is reported separately but not added to total_cost.
    total_cost = electricity_cost + demand_charge + deadline_penalty
    return {"electricity_cost": electricity_cost, "demand_charge": demand_charge,
            "opportunity_cost": opp_cost, "n_missed_deadlines": n_missed,
            "deadline_penalty": deadline_penalty, "total_cost": total_cost,
            "total_energy_kwh": float(total_draw.sum() * dt), "peak_kw": peak_kw}


def naive_forecast(df, date):
    date_ts = pd.Timestamp(date).normalize()
    start_lookback = date_ts - pd.Timedelta(days=10)
    mask  = (df["timestamp"] >= start_lookback) & (df["timestamp"] < date_ts)
    hist  = df.loc[mask]
    preds = np.zeros(24)
    for h in range(24):
        vals = hist.loc[hist["hour"] == h, "rt_lbmp_kwh"].values
        preds[h] = float(vals.mean()) if len(vals) > 0 else 0.0
    return preds


def get_hourly_prices(df, date):
    date_ts = pd.Timestamp(date).normalize()
    mask = ((df["timestamp"] >= date_ts) &
            (df["timestamp"] < date_ts + pd.Timedelta(days=1)))
    day_df = df.loc[mask].sort_values("timestamp")
    if len(day_df) < 24:
        return None
    return day_df["rt_lbmp_kwh"].values[:24].astype(float)

In [ ]:
# Verify the import works cleanly
from ev_charging_utils import (
    PARAMS, generate_daily_fleet_schedule, solve_charging_lp,
    fifo_heuristic, evaluate_schedule, naive_forecast, get_hourly_prices
)
print("ev_charging_utils imported successfully.")
print(f"  PARAMS keys     : {list(PARAMS.keys())}")
print(f"  Functions loaded: generate_daily_fleet_schedule, solve_charging_lp,")
print(f"                    fifo_heuristic, evaluate_schedule,")
print(f"                    naive_forecast, get_hourly_prices")

# Quick sanity check
import pandas as pd, json
_df = pd.read_csv("data/nyiso_prices_weather_nyc_2025.csv", parse_dates=["timestamp"])
_df = _df.sort_values("timestamp").reset_index(drop=True)
with open("data/nyc_fleet_profiles.json") as _f:
    _fp = json.load(_f)

_date = pd.Timestamp("2025-10-01")
_veh  = generate_daily_fleet_schedule(_fp, _date, params=PARAMS)
_p    = get_hourly_prices(_df, _date)
_res  = solve_charging_lp(_veh, _p, PARAMS, month=_date.month)
_ev   = evaluate_schedule(_res["schedule"], _p, _veh, _fp, _date, PARAMS)
print(f"\nSanity check (2025-10-01): {len(_veh)} vehicles, "
      f"total cost = ${_ev['total_cost']:.2f}, missed = {_ev['n_missed_deadlines']}")
print("All checks passed ✓")

## Next Steps

In **Notebook 1** (ML) and **Notebook 2** (DL), you will:
1. Train price forecasting models
2. Generate 24-hour-ahead price predictions for each validation day
3. Feed those predictions into `solve_charging_lp()` from this notebook
4. Evaluate with `evaluate_schedule()` against true prices
5. Compare your ML/DL cost to the naive and perfect foresight benchmarks

```python
from ev_charging_utils import solve_charging_lp, evaluate_schedule, get_hourly_prices, PARAMS

# For each validation day:
your_forecast = your_model.predict(features)           # array of 24 $/kWh
result        = solve_charging_lp(vehicles, your_forecast, PARAMS, month=date.month)
cost          = evaluate_schedule(result['schedule'], true_prices, vehicles,
                                  fleet_profiles, date, PARAMS)
```

The gap between your model's cost and the perfect foresight cost measures
how much money is left on the table due to forecast error.

| Benchmark | Typical cost | Goal |
|-----------|-------------|------|
| Perfect Foresight LP | $\approx$ ✓ | Lower bound |
| **Your ML/DL LP**    | $\approx$ ? | Minimize gap to PF |
| Naive Forecast LP    | $\approx$ ✓ | Beat this |
| FIFO Heuristic       | $\approx$ ✓ | Upper bound |